# THINGS CPU-Safe Colab Checks

This notebook is for Colab runtimes without GPU quota. It is intentionally CPU-safe: it restores data, verifies metadata, builds splits, and runs a tiny dataloader dry-run. It does not train ResNet-50, extract full embeddings, or run any long image pass.

Use this while waiting for GPU access to return.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/sabszh/human-things.git"
PROJECT_ROOT = Path("/content/human-things")

RESET_LOCAL_REPO = True
USE_DRIVE_DATA_ZIP = True
DRIVE_DATA_ZIP = Path("/content/drive/MyDrive/human-things-data/human-things-data.zip")
DRIVE_DATA_FILE_ID = "1OofSEPS34SA6Jol3OIqO208ekIHz1UEF"
LOCAL_DATA_ZIP = Path("/content/human-things-data.zip")

INSTALL_LIGHT_DEPS = True
ALLOW_OSF_FALLBACK = False  # Keep false for CPU-safe runs; OSF image download is slow and unnecessary if the Drive zip exists.
COPY_CHECK_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/human-things-runs")

print("CPU-safe configuration", flush=True)
print("- repo:", REPO_URL, flush=True)
print("- project root:", PROJECT_ROOT, flush=True)
print("- reset local repo:", RESET_LOCAL_REPO, flush=True)
print("- mounted Drive zip:", DRIVE_DATA_ZIP, flush=True)
print("- shared Drive file id:", DRIVE_DATA_FILE_ID, flush=True)
print("- local fallback zip:", LOCAL_DATA_ZIP, flush=True)
print("- install light deps:", INSTALL_LIGHT_DEPS, flush=True)
print("- OSF fallback enabled:", ALLOW_OSF_FALLBACK, flush=True)


## 1. Runtime Check

This notebook is expected to run on CPU. If CUDA is false, that is fine here.


In [ ]:
try:
    import torch
    print("Python:", sys.version, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("CUDA:", torch.cuda.is_available(), flush=True)
    print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", flush=True)
except Exception as exc:
    print("Torch check failed:", repr(exc), flush=True)


## 2. Mount Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped or failed:", repr(exc), flush=True)


## 3. Clone Repo


In [ ]:
def run_checked(command, cwd=None, check=True):
    print("\n$", " ".join(map(str, command)), flush=True)
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.stderr:
        print(result.stderr, flush=True)
    print("exit code:", result.returncode, flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, command))}")
    return result

print("Step 3: preparing repo checkout", flush=True)
os.chdir("/content")
print("Current working directory:", Path.cwd(), flush=True)
remote_check = run_checked(["git", "ls-remote", "--heads", REPO_URL], check=False)
if remote_check.returncode != 0:
    raise RuntimeError("Colab cannot access the GitHub repo. Read the git output above.")

if RESET_LOCAL_REPO:
    print(f"Removing temporary checkout: {PROJECT_ROOT}", flush=True)
    shutil.rmtree(PROJECT_ROOT, ignore_errors=True)

if not PROJECT_ROOT.exists():
    run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
else:
    pull = run_checked(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=False)
    if pull.returncode != 0:
        print("Pull failed; recreating checkout.", flush=True)
        shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
        run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd(), flush=True)
run_checked(["git", "rev-parse", "--short", "HEAD"])
run_checked(["find", "scripts", "-maxdepth", "1", "-type", "f", "-print"])


## 4. Install Light Dependencies

This avoids reinstalling torch. Colab CPU usually already has torch/torchvision. If torch is missing, use the GPU notebook or install PyTorch manually later.


In [ ]:
if INSTALL_LIGHT_DEPS:
    print("Installing light dependencies only: pandas numpy pillow scikit-learn tqdm gdown", flush=True)
    subprocess.run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "numpy",
        "Pillow",
        "scikit-learn",
        "tqdm",
        "gdown",
    ], check=True)
    print("Light dependencies installed.", flush=True)
else:
    print("Skipping dependency install.", flush=True)


## 5. Restore Data From Drive Zip

This cell prefers the mounted Drive zip. If it is missing, it downloads the shared file by ID. OSF fallback is disabled by default for CPU-safe runs.


In [ ]:
def restore_zip(zip_path):
    print(f"Restoring data from zip: {zip_path}", flush=True)
    print(f"Zip size GB: {zip_path.stat().st_size / 1e9:.2f}", flush=True)
    data_dir = PROJECT_ROOT / "data"
    if data_dir.exists():
        print(f"Removing existing data folder: {data_dir}", flush=True)
        shutil.rmtree(data_dir)
    print("Unzipping. This can take several minutes for ~28k image files.", flush=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.namelist()
        print(f"Zip entries: {len(members)}", flush=True)
        archive.extractall(PROJECT_ROOT)
    print("Restored:", data_dir, flush=True)
    print("Top-level data folders:", sorted(p.name for p in data_dir.iterdir()), flush=True)

print("Step 5: restoring data", flush=True)
zip_to_restore = None
if USE_DRIVE_DATA_ZIP and DRIVE_DATA_ZIP.exists():
    print("Found mounted Drive zip.", flush=True)
    zip_to_restore = DRIVE_DATA_ZIP
elif USE_DRIVE_DATA_ZIP and LOCAL_DATA_ZIP.exists():
    print("Found local Colab zip.", flush=True)
    zip_to_restore = LOCAL_DATA_ZIP
elif USE_DRIVE_DATA_ZIP and DRIVE_DATA_FILE_ID:
    print(f"Mounted Drive zip not found: {DRIVE_DATA_ZIP}", flush=True)
    print(f"Downloading shared Drive file to: {LOCAL_DATA_ZIP}", flush=True)
    subprocess.run([
        sys.executable,
        "-m",
        "gdown",
        "--id",
        DRIVE_DATA_FILE_ID,
        "-O",
        str(LOCAL_DATA_ZIP),
    ], check=True)
    zip_to_restore = LOCAL_DATA_ZIP

if zip_to_restore and zip_to_restore.exists():
    restore_zip(zip_to_restore)
elif ALLOW_OSF_FALLBACK:
    print("No zip found; running OSF fallback without image training.", flush=True)
    subprocess.run([sys.executable, "scripts/00_setup_things_data.py"], check=True)
else:
    raise FileNotFoundError(
        "No Drive zip available. Upload human-things-data.zip to Drive or make the shared file accessible."
    )
print("Data restore finished.", flush=True)


## 6. Verify Data Completeness


In [ ]:
import pandas as pd

print("Step 6: checking processed CSVs and image folders", flush=True)
concepts = pd.read_csv(PROJECT_ROOT / "data/processed/concepts.csv")
images = pd.read_csv(PROJECT_ROOT / "data/processed/images.csv")

things_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGS/object_images"
cc0_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0"
things_count = sum(1 for _ in things_dir.rglob("*.jpg")) if things_dir.exists() else 0
cc0_count = sum(1 for _ in cc0_dir.rglob("*.jpg")) if cc0_dir.exists() else 0

print("concepts rows:", len(concepts), flush=True)
print("concept_index unique:", concepts["concept_index"].nunique(), flush=True)
print("images rows:", len(images), flush=True)
print("image concept_index nulls:", int(images["concept_index"].isna().sum()), flush=True)
print("image concept_index unique:", images["concept_index"].nunique(), flush=True)
print("THINGS image dir exists:", things_dir.exists(), flush=True)
print("THINGSplus CC0 image dir exists:", cc0_dir.exists(), flush=True)
print("THINGS jpg count:", things_count, flush=True)
print("THINGSplus CC0 jpg count:", cc0_count, flush=True)

assert len(concepts) == 1854
assert concepts["concept_index"].nunique() == 1854
assert images["concept_index"].isna().sum() == 0
assert images["concept_index"].nunique() == 1854
assert things_count == 26107
print("Data completeness checks passed.", flush=True)


## 7. Build Metadata and Splits


In [ ]:
print("Step 7: building image metadata and splits", flush=True)
subprocess.run([sys.executable, "scripts/01_make_metadata_csv.py"], check=True)
subprocess.run([sys.executable, "scripts/02_make_image_splits.py"], check=True)

metadata = pd.read_csv(PROJECT_ROOT / "data/baseline/image_metadata.csv")
splits = pd.read_csv(PROJECT_ROOT / "data/baseline/image_splits.csv")
print("metadata rows:", len(metadata), flush=True)
print("metadata concepts:", metadata["concept_id"].nunique(), flush=True)
print("missing metadata images:", int((~metadata["image_exists"].astype(bool)).sum()), flush=True)
print("split rows:", len(splits), flush=True)
print("split counts:", splits["split"].value_counts().to_dict(), flush=True)
print("concepts per split:", splits.groupby("split")["concept_id"].nunique().to_dict(), flush=True)
assert metadata["concept_id"].nunique() == 1854
assert splits["concept_id"].nunique() == 1854
assert int((~splits["image_exists"].astype(bool)).sum()) == 0
print("Metadata and split checks passed.", flush=True)


## 8. Tiny Dataloader Dry-Run

This loads a few images through the transform path. It does not build ResNet and does not train.


In [ ]:
print("Step 8: running tiny dataloader dry-run", flush=True)
subprocess.run([
    sys.executable,
    "scripts/03_train_resnet50_image_only.py",
    "--dry-run",
    "--batch-size", "8",
    "--num-workers", "0",
], check=True)
print("Tiny dataloader dry-run passed.", flush=True)


## 9. Inspect Current Outputs

This only lists existing outputs. If no GPU training has completed, checkpoint files will be missing, which is expected.


In [ ]:
print("Step 9: inspecting outputs", flush=True)
paths = [
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "outputs/baseline_resnet50",
    PROJECT_ROOT / "outputs/baseline_resnet50/best_model.pt",
    PROJECT_ROOT / "outputs/baseline_resnet50/metrics.json",
    PROJECT_ROOT / "outputs/baseline_resnet50/training_log.csv",
    PROJECT_ROOT / "outputs/baseline_resnet50/embeddings/concept_embeddings.npy",
]
for path in paths:
    if path.exists():
        size = path.stat().st_size if path.is_file() else 0
        print(f"exists: {path} size={size}", flush=True)
    else:
        print(f"missing: {path}", flush=True)

if (PROJECT_ROOT / "outputs").exists():
    print("Output files:", flush=True)
    for path in sorted((PROJECT_ROOT / "outputs").rglob("*")):
        if path.is_file():
            print("-", path.relative_to(PROJECT_ROOT), flush=True)


## 10. Copy Check Outputs to Drive


In [ ]:
if COPY_CHECK_OUTPUTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
    print("Step 10: copying CPU-check outputs to Drive", flush=True)
    run_dir = DRIVE_OUTPUT_ROOT / "cpu_checks"
    run_dir.mkdir(parents=True, exist_ok=True)
    for name in ["outputs", "data/baseline", "data/processed"]:
        src = PROJECT_ROOT / name
        dst = run_dir / name
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"Copied {src} -> {dst}", flush=True)
    print("Drive copy finished:", run_dir, flush=True)
else:
    print("Skipping Drive copy", flush=True)
